# The Standard Model in FeynPy — an `SM.fr`-style implementation

This is a literate FeynPy model file. It implements the same Standard Model
content as FeynRules `SM.fr`, but uses Python declarations throughout.

1. declare indices and parameters;
2. declare gauge-basis and physical fields;
3. declare gauge groups;
4. write the gauge-basis Lagrangian;
5. define the gauge-to-physical field transformations;
6. compile, inspect, and extract Feynman rules.


## 1. Setup and notation


In [1]:
import json
import re
import sys
import textwrap
from collections import Counter
from dataclasses import replace
from fractions import Fraction
from pathlib import Path
from types import SimpleNamespace

START = Path.cwd().resolve()
ROOT = next(
    (
        candidate
        for candidate in (START, *START.parents)
        if (candidate / "src").is_dir() and (candidate / "models" / "SM").is_dir()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Could not locate the FeynPy repository root")
MODEL_DIR = ROOT / "models" / "SM"
SRC = ROOT / "src"
for entry in (ROOT, SRC):
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))

from symbolica import Expression, S

from feynpy import (
    COLOR_ADJ_INDEX,
    COLOR_FUND_INDEX,
    WEAK_ADJ_INDEX,
    WEAK_FUND_INDEX,
    SPINOR_INDEX,
    LORENTZ_INDEX,
    CompiledLagrangian,
    DC,
    Field,
    FS,
    FieldTransformation,
    Gamma,
    GaugeFixing,
    GaugeGroup,
    GaugeRepresentation,
    GhostLagrangian,
    Model,
    Parameter,
    PartialD,
    ProjM,
    ProjP,
    flavor_index,
    rotation,
)
from symbolic.spenso_structures import (
    gauge_generator,
    structure_constant,
    weak_eps2,
    weak_gauge_generator,
    weak_structure_constant,
)
from symbolic.vertex_engine import I
from models.SM import standard_model_weak_tensor_components
from models.SM.SM_support import (
    electroweak_rxi_gauge_fixing_lagrangian,
    electroweak_scalar_ghost_lagrangian,
)

num = Expression.num
ONE = num(1)
TWO = num(2)
THREE = num(3)
FOUR = num(4)
SIX = num(6)
HALF = ONE / TWO
INV_SQRT2 = HALF**HALF

ANSI = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")
DISPLAY_PREFIX = re.compile(r"(?:python|spenso(?:_python)?)::(?:\{[^}]*\}::)?")
DISPLAY_TAG = re.compile(r"\{[^}]*\}::")


def clean(value):
    """Remove display namespaces without changing the symbolic expression."""
    if hasattr(value, "to_symbolica") and not hasattr(value, "to_canonical_string"):
        value = value.to_symbolica()
    rendered = (
        value.to_canonical_string()
        if hasattr(value, "to_canonical_string")
        else str(value)
    )
    rendered = ANSI.sub("", rendered)
    rendered = DISPLAY_PREFIX.sub("", rendered)
    return DISPLAY_TAG.sub("", rendered)


def print_rule(label, expression, width=116):
    print(label)
    lines = textwrap.wrap(
        clean(expression),
        width=width,
        break_long_words=False,
        break_on_hyphens=False,
    ) or [""]
    for line in lines:
        print("  " + line)


INCLUDE_GHOSTS = True
INCLUDE_GAUGE_FIXING = False
MODEL_NAME = "Standard Model"

print("Repository root       :", ROOT)
print("Include ghosts        :", INCLUDE_GHOSTS)
print("Include gauge fixing  :", INCLUDE_GAUGE_FIXING)


Repository root       : /Users/rems/Documents/MScThesis
Include ghosts        : True
Include gauge fixing  : False


### Correspondence with FeynRules

| FeynRules `SM.fr` object | FeynPy object used here |
|---|---|
| `IndexRange[Index[Generation]]` | `flavor_index(...)` |
| `M$Parameters` | `Parameter(...)` |
| `M$ClassesDescription` | `Field(...)` |
| `M$GaugeGroups` | `GaugeGroup(...)` |
| `FS[...]`, `DC[...]` | `FS(...)`, `DC(...)` |
| `LGauge + LFermions + ...` | the same named Python expressions |
| field `Definitions` | `FieldTransformation(...)` |
| `FeynmanRules[LSM]` | `L.feynman_rule(...)` |

The two syntaxes are different; the model-building stages are intentionally
kept parallel. The descriptive aliases `FieldStrength(...)` and `CovD(...)`
still exist, but this notebook uses the shorter `FS(...)` and `DC(...)`
spellings to stay closer to `SM.fr`.


## 2. Indices

In [2]:
Generation = flavor_index("Generation", 3, prefix="fl")

print("generation index:", Generation.name)
print("generation range:", Generation.dimension)
print("built-in weak indices:", WEAK_FUND_INDEX.name, WEAK_ADJ_INDEX.name)
print("built-in color indices:", COLOR_FUND_INDEX.name, COLOR_ADJ_INDEX.name)


generation index: Generation
generation range: 3
built-in weak indices: WeakFund WeakAdj
built-in color indices: ColorFund ColorAdj


## 3. Parameters (`M$Parameters`)

The Python variable names `g2` and `g3` represent the symbols `gw` and `gs`,
matching the FeynRules export. FeynPy compiles the gauge-basis theory using
`g1` and `gw`, then rewrites them to the physical electroweak basis
`g1 = ee/cw`, `gw = ee/sw`.

Yukawa and CKM objects carry explicit generation indices. Their component
tables are used when flavor classes are expanded to `e, mu, ...`.


In [3]:
def diagonal_components(prefix):
    return {
        (row, column): (S(f"{prefix}{row}") if row == column else num(0))
        for row in range(1, 4)
        for column in range(1, 4)
    }


def ckm_components():
    return {
        (row, column): S(f"CKM{row}{column}")
        for row in range(1, 4)
        for column in range(1, 4)
    }


def ckm_dagger_components():
    return {
        (column, row): S(f"CKMConj{row}{column}")
        for row in range(1, 4)
        for column in range(1, 4)
    }


g1 = Parameter("g1")
g2 = Parameter("gw")
g3 = Parameter("gs")
lam = Parameter("lam")
vev = Parameter("vev")
sw = Parameter("sw")
cw = Parameter("cw")
ee = Parameter("ee")

MW = Parameter("MW", value=g2.symbol * vev.symbol / TWO)
MZ = Parameter(
    "MZ",
    value=(g1.symbol**2 + g2.symbol**2)**HALF * vev.symbol / TWO,
)
MH = Parameter("MH", value=(TWO * lam.symbol * vev.symbol**2)**HALF)

xiA = Parameter("xiA", internal=False, value=1)
xiZ = Parameter("xiZ", internal=False, value=1)
xiW = Parameter("xiW", internal=False, value=1)
xiG = Parameter("xiG", internal=False, value=1)

Mvl = Parameter(
    "Mvl", indices=(Generation,),
    components={(i,): num(0) for i in range(1, 4)},
)
Ml = Parameter(
    "Ml", indices=(Generation,),
    components={(i,): INV_SQRT2 * vev.symbol * S(f"ye{i}") for i in range(1, 4)},
)
Mu = Parameter(
    "Mu", indices=(Generation,),
    components={(i,): INV_SQRT2 * vev.symbol * S(f"yu{i}") for i in range(1, 4)},
)
Md = Parameter(
    "Md", indices=(Generation,),
    components={(i,): INV_SQRT2 * vev.symbol * S(f"yd{i}") for i in range(1, 4)},
)

Yu = Parameter("yu", indices=(Generation, Generation), complex_param=True,
               components=diagonal_components("yu"))
YuDag = Parameter("YuDag", indices=(Generation, Generation), complex_param=True,
                  components=diagonal_components("yu"))
Yd = Parameter("yd", indices=(Generation, Generation), complex_param=True,
               components=diagonal_components("yd"))
YdDag = Parameter("YdDag", indices=(Generation, Generation), complex_param=True,
                  components=diagonal_components("yd"))
Ye = Parameter("yl", indices=(Generation, Generation), complex_param=True,
               components=diagonal_components("ye"))
YeDag = Parameter("YeDag", indices=(Generation, Generation), complex_param=True,
                  components=diagonal_components("ye"))
CKM = Parameter(
    "CKM", indices=(Generation, Generation), complex_param=True,
    components=ckm_components(), unitary_partner="CKMDag",
)
CKMDag = Parameter(
    "CKMDag", indices=(Generation, Generation), complex_param=True,
    components=ckm_dagger_components(), unitary_partner="CKM",
)

parameters = SimpleNamespace(
    g1=g1, g2=g2, g3=g3, lam=lam, vev=vev,
    Mvl=Mvl, Ml=Ml, Mu=Mu, Md=Md, MW=MW, MZ=MZ, MH=MH,
    sw=sw, cw=cw, ee=ee, xiA=xiA, xiZ=xiZ, xiW=xiW, xiG=xiG,
    Yu=Yu, YuDag=YuDag, Yd=Yd, YdDag=YdDag,
    Ye=Ye, YeDag=YeDag, CKM=CKM, CKMDag=CKMDag,
)
all_parameters = tuple(parameters.__dict__.values())

print(f"{'attribute':<10} {'symbol':<10} {'rank':>4}  definition/sample")
print("-" * 72)
for attribute, parameter in parameters.__dict__.items():
    value = clean(parameter.value) if parameter.value is not None else "symbolic"
    print(f"{attribute:<10} {clean(parameter.symbol):<10} {len(parameter.indices):>4}  {value}")


attribute  symbol     rank  definition/sample
------------------------------------------------------------------------
g1         g1            0  symbolic
g2         gw            0  symbolic
g3         gs            0  symbolic
lam        lam           0  symbolic
vev        vev           0  symbolic
Mvl        Mvl           1  symbolic
Ml         Ml            1  symbolic
Mu         Mu            1  symbolic
Md         Md            1  symbolic
MW         MW            0  1/2*gw*vev
MZ         MZ            0  (g1^2+gw^2)^(1/2)*1/2*vev
MH         MH            0  (2*lam*vev^2)^(1/2)
sw         sw            0  symbolic
cw         cw            0  symbolic
ee         ee            0  symbolic
xiA        xiA           0  1
xiZ        xiZ           0  1
xiW        xiW           0  1
xiG        xiG           0  1
Yu         yu            2  symbolic
YuDag      YuDag         2  symbolic
Yd         yd            2  symbolic
YdDag      YdDag         2  symbolic
Ye         yl            2  

## 4. Particle classes (`M$ClassesDescription`)



In [4]:
# Gauge-basis gauge fields and ghosts.
B = Field("B", spin=1, self_conjugate=True, indices=(LORENTZ_INDEX,))
Wi = Field(
    "Wi", spin=1, self_conjugate=True,
    indices=(LORENTZ_INDEX, WEAK_ADJ_INDEX),
)
G = Field(
    "G", spin=1, self_conjugate=True,
    indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX), mass=num(0),
)
ghB = Field(
    "ghB", spin=0, kind="ghost", self_conjugate=False,
    ghost_of=B, conjugate_symbol=S("ghBbar"),
)
ghWi = Field(
    "ghWi", spin=0, kind="ghost", self_conjugate=False,
    indices=(WEAK_ADJ_INDEX,), ghost_of=Wi,
    conjugate_symbol=S("ghWibar"),
)
ghG = Field(
    "ghG", spin=0, kind="ghost", self_conjugate=False,
    indices=(COLOR_ADJ_INDEX,), ghost_of=G,
    conjugate_symbol=S("ghGbar"), mass=num(0),
    quantum_numbers={"GhostNumber": ONE},
)

# Gauge-basis matter and Higgs fields.
LL = Field(
    "LL", spin=Fraction(1, 2), self_conjugate=False,
    indices=(SPINOR_INDEX, WEAK_FUND_INDEX, Generation),
    quantum_numbers={"Y": -HALF},
)
lR = Field(
    "lR", spin=Fraction(1, 2), self_conjugate=False,
    indices=(SPINOR_INDEX, Generation), quantum_numbers={"Y": -ONE},
)
QL = Field(
    "QL", spin=Fraction(1, 2), self_conjugate=False,
    indices=(SPINOR_INDEX, WEAK_FUND_INDEX, Generation, COLOR_FUND_INDEX),
    quantum_numbers={"Y": ONE / SIX},
)
uR = Field(
    "uR", spin=Fraction(1, 2), self_conjugate=False,
    indices=(SPINOR_INDEX, Generation, COLOR_FUND_INDEX),
    quantum_numbers={"Y": TWO / THREE},
)
dR = Field(
    "dR", spin=Fraction(1, 2), self_conjugate=False,
    indices=(SPINOR_INDEX, Generation, COLOR_FUND_INDEX),
    quantum_numbers={"Y": -(ONE / THREE)},
)
Phi = Field(
    "Phi", spin=0, self_conjugate=False,
    symbol=S("Phi"), conjugate_symbol=S("Phibar"),
    indices=(WEAK_FUND_INDEX,), quantum_numbers={"Y": HALF},
)

source_fields = (LL, lR, QL, uR, dR, Phi, B, Wi, G, ghB, ghWi, ghG)

print("Gauge-basis fields:")
print([field.name for field in source_fields])


Gauge-basis fields:
['LL', 'lR', 'QL', 'uR', 'dR', 'Phi', 'B', 'Wi', 'G', 'ghB', 'ghWi', 'ghG']


In [5]:
# Physical vector and scalar fields.
W = Field(
    "W", spin=1, self_conjugate=False, conjugate_symbol=S("Wbar"),
    indices=(LORENTZ_INDEX,), mass=MW.symbol, quantum_numbers={"Q": ONE},
)
Z = Field(
    "Z", spin=1, self_conjugate=True, indices=(LORENTZ_INDEX,),
    mass=MZ.symbol, quantum_numbers={"Q": num(0)},
)
A = Field(
    "A", spin=1, self_conjugate=True, indices=(LORENTZ_INDEX,),
    mass=num(0), quantum_numbers={"Q": num(0)},
)
H = Field(
    "H", spin=0, self_conjugate=True,
    mass=MH.symbol, quantum_numbers={"Q": num(0)},
)

z_unphysical_mass = (xiZ.value * MZ.symbol**2)**HALF if INCLUDE_GAUGE_FIXING else MZ.symbol
w_unphysical_mass = (xiW.value * MW.symbol**2)**HALF if INCLUDE_GAUGE_FIXING else MW.symbol
G0 = Field(
    "G0", spin=0, self_conjugate=True, mass=z_unphysical_mass,
    quantum_numbers={"Q": num(0)}, goldstone_of=Z,
)
GP = Field(
    "GP", spin=0, self_conjugate=False, conjugate_symbol=S("GPbar"),
    mass=w_unphysical_mass, quantum_numbers={"Q": ONE}, goldstone_of=W,
)

# Physical fermion flavor classes.
vl = Field(
    "vl", spin=Fraction(1, 2), self_conjugate=False,
    indices=(SPINOR_INDEX, Generation), mass=Mvl,
    quantum_numbers={"Q": num(0), "LeptonNumber": ONE},
    flavor_index=Generation, class_members=("ve", "vm", "vt"),
)
l = Field(
    "l", spin=Fraction(1, 2), self_conjugate=False,
    indices=(SPINOR_INDEX, Generation), mass=Ml,
    quantum_numbers={"Q": -ONE, "LeptonNumber": ONE},
    flavor_index=Generation, class_members=("e", "mu", "ta"),
)
uq = Field(
    "uq", spin=Fraction(1, 2), self_conjugate=False,
    indices=(SPINOR_INDEX, Generation, COLOR_FUND_INDEX), mass=Mu,
    quantum_numbers={"Q": TWO / THREE},
    flavor_index=Generation, class_members=("u", "c", "t"),
)
dq = Field(
    "dq", spin=Fraction(1, 2), self_conjugate=False,
    indices=(SPINOR_INDEX, Generation, COLOR_FUND_INDEX), mass=Md,
    quantum_numbers={"Q": -(ONE / THREE)},
    flavor_index=Generation, class_members=("d", "s", "b"),
)

# Physical ghosts. A barred ghost is an independent antighost field.
ghA = Field(
    "ghA", spin=0, kind="ghost", self_conjugate=False,
    ghost_of=A, conjugate_symbol=S("ghAbar"), mass=num(0),
    quantum_numbers={"GhostNumber": ONE},
)
ghZ = Field(
    "ghZ", spin=0, kind="ghost", self_conjugate=False,
    ghost_of=Z, conjugate_symbol=S("ghZbar"), mass=z_unphysical_mass,
    quantum_numbers={"GhostNumber": ONE},
)
ghWp = Field(
    "ghWp", spin=0, kind="ghost", self_conjugate=False,
    ghost_of=W, conjugate_symbol=S("ghWpbar"), mass=w_unphysical_mass,
    quantum_numbers={"GhostNumber": ONE, "Q": ONE},
)
ghWm = Field(
    "ghWm", spin=0, kind="ghost", self_conjugate=False,
    ghost_of="Wbar", conjugate_symbol=S("ghWmbar"), mass=w_unphysical_mass,
    quantum_numbers={"GhostNumber": ONE, "Q": -ONE},
)

fields = SimpleNamespace(
    LL=LL, lR=lR, QL=QL, uR=uR, dR=dR, Phi=Phi,
    B=B, Wi=Wi, ghB=ghB, ghWi=ghWi,
    vl=vl, l=l, uq=uq, dq=dq, H=H, G0=G0, GP=GP,
    W=W, Z=Z, A=A, G=G,
    ghA=ghA, ghZ=ghZ, ghWp=ghWp, ghWm=ghWm, ghG=ghG,
)
physical_fields = (vl, l, uq, dq, H, G0, GP, W, Z, A, G)
if INCLUDE_GHOSTS:
    physical_fields += (ghA, ghZ, ghWp, ghWm, ghG)

print("Physical field classes:")
print([field.name for field in physical_fields])
print("charged leptons:", [member.name for member in l.class_members])
print("up quarks      :", [member.name for member in uq.class_members])
print("down quarks    :", [member.name for member in dq.class_members])


Physical field classes:
['vl', 'l', 'uq', 'dq', 'H', 'G0', 'GP', 'W', 'Z', 'A', 'G', 'ghA', 'ghZ', 'ghWp', 'ghWm', 'ghG']
charged leptons: ['e', 'mu', 'ta']
up quarks      : ['u', 'c', 't']
down quarks    : ['d', 's', 'b']


## 5. Gauge groups (`M$GaugeGroups`)




In [6]:
U1Y = GaugeGroup(
    "U1Y", abelian=True, coupling=g1, gauge_boson=B,
    ghost_field=ghB, charge="Y",
)
SU2L = GaugeGroup(
    "SU2L", abelian=False, coupling=g2, gauge_boson=Wi,
    ghost_field=ghWi, structure_constant=weak_structure_constant,
    representations=(
        GaugeRepresentation(
            index=WEAK_FUND_INDEX,
            generator_builder=weak_gauge_generator,
            name="doublet",
        ),
    ),
)
SU3C = GaugeGroup(
    "SU3C", abelian=False, coupling=g3, gauge_boson=G,
    ghost_field=ghG, structure_constant=structure_constant,
    representations=(
        GaugeRepresentation(
            index=COLOR_FUND_INDEX,
            generator_builder=gauge_generator,
            name="fundamental",
        ),
    ),
)
gauge_groups = (U1Y, SU2L, SU3C)

print(f"{'group':<8} {'type':<12} {'coupling':<10} gauge field")
print("-" * 48)
for group in gauge_groups:
    print(
        f"{group.name:<8} "
        f"{('abelian' if group.abelian else 'non-abelian'):<12} "
        f"{clean(group.coupling.symbol):<10} {group.gauge_boson.name}"
    )


group    type         coupling   gauge field
------------------------------------------------
U1Y      abelian      g1         B
SU2L     non-abelian  gw         Wi
SU3C     non-abelian  gs         G


## 6. Gauge-basis Lagrangian




In [7]:
mu, nu = S("mu"), S("nu")
weak_adj, colour_adj = S("aW"), S("aC")
spinor, weak_left, weak_right, colour = S("sp"), S("ii"), S("jj"), S("cc")
f1, f2, f3 = S("ff1"), S("ff2"), S("ff3")

LGauge = (
    -ONE / FOUR * FS(U1Y, mu, nu) * FS(U1Y, mu, nu)
    - ONE / FOUR * FS(SU2L, mu, nu, weak_adj) * FS(SU2L, mu, nu, weak_adj)
    - ONE / FOUR * FS(SU3C, mu, nu, colour_adj) * FS(SU3C, mu, nu, colour_adj)
)

LFermions = (
    I * QL.bar * Gamma(mu) * DC(QL, mu)
    + I * LL.bar * Gamma(mu) * DC(LL, mu)
    + I * uR.bar * Gamma(mu) * DC(uR, mu)
    + I * dR.bar * Gamma(mu) * DC(dR, mu)
    + I * lR.bar * Gamma(mu) * DC(lR, mu)
)

LHiggs = (
    DC(Phi.bar, mu) * DC(Phi, mu)
    + lam * vev**2 * Phi.bar * Phi
    - lam * Phi.bar * Phi * Phi.bar * Phi
)

LYukawa = (
    -Yd(f2, f3) * CKM(f1, f2)
    * QL.bar(spinor, weak_left, f1, colour)
    * dR(spinor, f3, colour) * Phi(weak_left)

    -Ye(f1, f3) * LL.bar(spinor, weak_left, f1)
    * lR(spinor, f3) * Phi(weak_left)

    -Yu(f1, f2) * QL.bar(spinor, weak_left, f1, colour)
    * uR(spinor, f2, colour) * Phi.bar(weak_right)
    * weak_eps2(weak_left, weak_right)

    -YdDag(f3, f2) * CKMDag(f2, f1) * Phi.bar(weak_left)
    * dR.bar(spinor, f3, colour) * QL(spinor, weak_left, f1, colour)

    -YeDag(f3, f1) * Phi.bar(weak_left)
    * lR.bar(spinor, f3) * LL(spinor, weak_left, f1)

    -YuDag(f2, f1) * weak_eps2(weak_left, weak_right)
    * Phi(weak_right) * uR.bar(spinor, f2, colour)
    * QL(spinor, weak_left, f1, colour)
)


### Gauge-fixing and ghost sectors

In [8]:
# Start with the four physical Standard Model sectors.
LSM = LGauge + LFermions + LHiggs + LYukawa

# Optional sectors are ordinary expressions too. Add them only when enabled;
# no explicit Lagrangian container is needed in user code.
if INCLUDE_GAUGE_FIXING:
    LGaugeFixing = (
        electroweak_rxi_gauge_fixing_lagrangian(fields, parameters)
        + GaugeFixing(SU3C, xi=xiG.value)
    )
    LSM += LGaugeFixing

if INCLUDE_GHOSTS:
    LGhost = (
        PartialD(ghB.bar, mu) * PartialD(ghB, mu)
        + GhostLagrangian(SU2L)
        + GhostLagrangian(SU3C)
        + electroweak_scalar_ghost_lagrangian(fields, parameters)
    )
    LSM += LGhost

print("Gauge fixing included:", INCLUDE_GAUGE_FIXING)
print("Ghost sector included:", INCLUDE_GHOSTS)
print("Complete LSM expression assembled")


# -----------------------------------------------------------------------------
# Manual alternative 
# -----------------------------------------------------------------------------
# mu_gf_1, mu_gf_2, a_gf = S("mu_gf_1"), S("mu_gf_2"), S("a_gf")
# LGaugeFixingQCD_manual = (
#     -ONE / (TWO * xiG.value)
#     * PartialD(G(mu_gf_1, a_gf), mu_gf_1)
#     * PartialD(G(mu_gf_2, a_gf), mu_gf_2)
# )
#
# div_A = PartialD(A(S("mu_A")), S("mu_A"))
# div_Z = PartialD(Z(S("mu_Z")), S("mu_Z"))
# LGaugeFixingAZ_manual = (
#     -ONE / (TWO * xiA.value) * div_A * div_A
#     -ONE / (TWO * xiZ.value) * div_Z * div_Z
#     + MZ.symbol * div_Z * G0
#     - xiZ.value * MZ.symbol**2 / TWO * G0 * G0
# )
#
# mu_gh = S("mu_gh")
# aw = S("aw")
# aw_bar, aw_gauge, aw_ghost = S("aw_bar"), S("aw_gauge"), S("aw_ghost")
# ac = S("ac")
# ac_bar, ac_gauge, ac_ghost = S("ac_bar"), S("ac_gauge"), S("ac_ghost")
#
# LGhostOrdinary_manual = (
#     PartialD(ghB.bar, mu_gh) * PartialD(ghB, mu_gh)
#
#     # SU(2): kinetic and antighost-gauge-ghost terms.
#     + PartialD(ghWi.bar(aw), mu_gh) * PartialD(ghWi(aw), mu_gh)
#     - g2 * weak_structure_constant(aw_bar, aw_gauge, aw_ghost)
#     * PartialD(ghWi.bar(aw_bar), mu_gh)
#     * Wi(mu_gh, aw_gauge)
#     * ghWi(aw_ghost)
#
#     # SU(3): kinetic and antighost-gluon-ghost terms.
#     + PartialD(ghG.bar(ac), mu_gh) * PartialD(ghG(ac), mu_gh)
#     - g3 * structure_constant(ac_bar, ac_gauge, ac_ghost)
#     * PartialD(ghG.bar(ac_bar), mu_gh)
#     * G(mu_gh, ac_gauge)
#     * ghG(ac_ghost)
# )
#
#   L_ghost,Phi = -cbar^A xi_AB [
#       (T^B v)^dagger T^C Phi + (T^C Phi)^dagger T^B v
#   ] c^C.
#
# zero = num(0)
# ew_generators = (
#     ((-I * g1.symbol * HALF, zero), (zero, -I * g1.symbol * HALF)),
#     ((zero, -I * g2.symbol * HALF), (-I * g2.symbol * HALF, zero)),
#     ((zero, -g2.symbol * HALF), (g2.symbol * HALF, zero)),
#     ((-I * g2.symbol * HALF, zero), (zero, I * g2.symbol * HALF)),
# )
# ew_vacuum = (zero, vev.symbol * INV_SQRT2)
# ew_vacuum_images = tuple(
#     tuple(
#         sum(
#             (matrix[row][column] * ew_vacuum[column] for column in range(2)),
#             zero,
#         )
#         for row in range(2)
#     )
#     for matrix in ew_generators
# )
# ew_ghosts = (ghB, ghWi(num(1)), ghWi(num(2)), ghWi(num(3)))
# ew_antighosts = (
#     ghB.bar,
#     ghWi.bar(num(1)),
#     ghWi.bar(num(2)),
#     ghWi.bar(num(3)),
# )
#
# def real_conjugate_manual(value):
#     result = value.conj()
#     for symbol in (g1.symbol, g2.symbol, vev.symbol):
#         result = result.replace(symbol.conj(), symbol)
#     return result
#
# LGhostScalar_manual = zero
# for left in range(4):
#     for right in range(4):
#         for component in range(2):
#             phi_coefficient = -sum(
#                 (
#                     real_conjugate_manual(ew_vacuum_images[left][row])
#                     * ew_generators[right][row][component]
#                     for row in range(2)
#                 ),
#                 zero,
#             )
#             phibar_coefficient = -sum(
#                 (
#                     real_conjugate_manual(ew_generators[right][row][component])
#                     * ew_vacuum_images[left][row]
#                     for row in range(2)
#                 ),
#                 zero,
#             )
#             if phi_coefficient.expand().to_canonical_string() != "0":
#                 LGhostScalar_manual += (
#                     phi_coefficient
#                     * ew_antighosts[left]
#                     * ew_ghosts[right]
#                     * Phi(num(component + 1))
#                 )
#             if phibar_coefficient.expand().to_canonical_string() != "0":
#                 LGhostScalar_manual += (
#                     phibar_coefficient
#                     * ew_antighosts[left]
#                     * ew_ghosts[right]
#                     * Phi.bar(num(component + 1))
#                 )
#
# LGhost_manual = LGhostOrdinary_manual + LGhostScalar_manual


Gauge fixing included: False
Ghost sector included: True
Complete LSM expression assembled


## 7. Field definitions: gauge basis to physical basis

These are the FeynPy equivalent of the `Definitions` attached to field
classes in `SM.fr`:

- `B` and `Wi` mix into `A`, `Z`, and charged `W` fields;
- `Phi` is shifted by the vacuum expectation value and split into Higgs and
  Goldstone fields;
- left- and right-handed fermions receive `ProjM` and `ProjP`;
- the lower quark doublet component carries the CKM rotation;
- electroweak ghosts follow the same neutral/charged mixing pattern.

`components={slot: value}` restricts a definition to one multiplet component.
All definitions are applied simultaneously in one transformation stage.

`rotation(CKM, CKMDag)` declares how the CKM matrix acts on the generation
index. In `QL[2] -> CKM * ProjM * dq`, an unbarred field receives
`CKM[i,j]`, while the automatically generated barred transformation receives
the Hermitian-conjugate entry `CKMDag[j,i]`. It is transformation metadata,
not a numerical matrix multiplication performed at this line.


In [9]:
ckm_rotation = rotation(CKM, CKMDag)

transformations = (
    FieldTransformation(B, -sw.symbol * Z + cw.symbol * A),
    FieldTransformation(Wi, INV_SQRT2 * W.bar + INV_SQRT2 * W, components={1: 1}),
    FieldTransformation(Wi, INV_SQRT2 / I * W.bar - INV_SQRT2 / I * W, components={1: 2}),
    FieldTransformation(Wi, cw.symbol * Z + sw.symbol * A, components={1: 3}),

    FieldTransformation(Phi, -I * GP, components={0: 1}),
    FieldTransformation(
        Phi,
        vev.symbol * INV_SQRT2 + INV_SQRT2 * H + I * INV_SQRT2 * G0,
        components={0: 2},
    ),

    FieldTransformation(LL, ProjM * vl, components={1: 1}),
    FieldTransformation(LL, ProjM * l, components={1: 2}),
    FieldTransformation(lR, ProjP * l),
    FieldTransformation(QL, ProjM * uq, components={1: 1}),
    FieldTransformation(QL, ckm_rotation * ProjM * dq, components={1: 2}),
    FieldTransformation(uR, ProjP * uq),
    FieldTransformation(dR, ProjP * dq),

    FieldTransformation(ghB, -sw.symbol * ghZ + cw.symbol * ghA),
    FieldTransformation(ghWi, INV_SQRT2 * ghWp + INV_SQRT2 * ghWm, components={0: 1}),
    FieldTransformation(ghWi, -INV_SQRT2 / I * ghWp + INV_SQRT2 / I * ghWm, components={0: 2}),
    FieldTransformation(ghWi, cw.symbol * ghZ + sw.symbol * ghA, components={0: 3}),
)

print(f"{'source':<8} {'components':<20} replacement")
print("-" * 100)
for transformation in transformations:
    components = str(dict(transformation.components)) if transformation.components else "all"
    print(f"{transformation.source.name:<8} {components:<20} {clean(transformation.expr)}")
print("total definitions:", len(transformations))


source   components           replacement
----------------------------------------------------------------------------------------------------
B        all                  -sw * Z + cw * A
Wi       {1: 1}               (1/2)^(1/2) * W.bar + (1/2)^(1/2) * W
Wi       {1: 2}               -1𝑖*(1/2)^(1/2) * W.bar + 1𝑖*(1/2)^(1/2) * W
Wi       {1: 3}               cw * Z + sw * A
Phi      {0: 1}               -1𝑖 * GP
Phi      {0: 2}               (1/2)^(1/2)*vev + (1/2)^(1/2) * H + 1𝑖*(1/2)^(1/2) * G0
LL       {1: 1}               ProjM * vl
LL       {1: 2}               ProjM * l
lR       all                  ProjP * l
QL       {1: 1}               ProjM * uq
QL       {1: 2}               CKM * ProjM * dq
uR       all                  ProjP * uq
dR       all                  ProjP * dq
ghB      all                  -sw * ghZ + cw * ghA
ghWi     {0: 1}               (1/2)^(1/2) * ghWp + (1/2)^(1/2) * ghWm
ghWi     {0: 2}               1𝑖*(1/2)^(1/2) * ghWp + -1𝑖*(1/2)^(1/2) * ghWm
ghWi   

## 8. Compile the model

One source `Model` is sufficient. FeynRules performs the following operations
after loading a model file; FeynPy keeps them visible:

1. `Model(LSM, ...)` associates the complete expression with its fields,
   parameters, and gauge groups;
2. `source_model.lagrangian()` expands the declared `DC(...)` and `FS(...)` factors;
3. `expand_index_components(...)` unfolds the finite SU(2) indices;
4. `transform_fields(...)` applies the physical-field definitions;
5. `simplify_parameter_identities()` reduces CKM/unitarity identities;
6. the final small helper rewrites `g1` and `gw` as `ee/cw` and `ee/sw`.

The result `L` is already a compiled physical-basis Lagrangian and directly
provides `L.feynman_rule(...)`. There is no need to construct a temporary
`Model` for every sector.


In [10]:
#Ta=1/2 σa​ --- structure constant f=epsilon_abc --- antisymmetric SU(2) doublet tensor eps_ij,
weak_tensor_components = standard_model_weak_tensor_components()

real_symbols = (
    g1, g2, g3, lam, vev, MW, MZ, MH,
    xiA, xiZ, xiW, xiG, sw, cw, ee,
    xiA.value, xiZ.value, xiW.value, xiG.value,
)


def express_in_physical_couplings(lagrangian):
    """Rewrite the unbroken electroweak couplings in the SM.fr output basis."""
    substitutions = (
        (g1.symbol, ee.symbol / cw.symbol),
        (g2.symbol, ee.symbol / sw.symbol),
    )
    terms = []
    for term in lagrangian.terms:
        coupling = term.coupling
        for symbol, definition in substitutions:
            coupling = coupling.replace(symbol, definition)
        terms.append(replace(term, coupling=coupling.cancel().expand()))
    return CompiledLagrangian(terms=tuple(terms), parameters=all_parameters)


source_model = Model(
    LSM,
    name=f"{MODEL_NAME} gauge basis",
    gauge_groups=gauge_groups,
    fields=source_fields,
    parameters=all_parameters,
)

L_gauge_basis = source_model.lagrangian()

L_weak_components = L_gauge_basis.expand_index_components(
    WEAK_FUND_INDEX,
    WEAK_ADJ_INDEX,
    tensor_components=weak_tensor_components,
)
L_physical = (
    L_weak_components
    .transform_fields(
        *transformations,
        repeat=False,
        real_symbols=real_symbols,
    )
    .simplify_parameter_identities()
)
L = express_in_physical_couplings(L_physical)
print("gauge-basis terms :", L_gauge_basis.to_symbolica())
print("physical terms :", L_weak_components.to_symbolica())
print("compiled terms :", L_physical.to_symbolica())
print("final terms :", L.to_symbolica())

source_report = source_model.validate()
compiled_report = L.validate()
arity_counts = Counter(signature.arity for signature in L.vertex_signatures(flavor_expand=True))

print("gauge-basis compiled terms :", len(L_gauge_basis.terms))
print("weak-component terms       :", len(L_weak_components.terms))
print("physical compiled terms    :", len(L.terms))
print("compact vertex signatures :", len(L.vertex_signatures()))
print("expanded arity counts      :", dict(sorted(arity_counts.items())))
print("source model validation    :", source_report.ok)
print("compiled validation        :", compiled_report.ok)

assert source_report.ok
assert compiled_report.ok
assert arity_counts == {2: 30, 3: 139, 4: 32}


gauge-basis terms : 1/2*g1*gw*g(mink(4, mu),mink(4, nu))*t(coad(3, weak_adj_Wi_SU2L_mix),cof(2, w_bar_Phi),cof(2, w_Phi))*B(mu)*Wi(nu,weak_adj_Wi_SU2L_mix)*Phi(w_Phi)*Phibar(w_bar_Phi)+1/2*g1*gw*g(mink(4, mu),mink(4, nu))*t(coad(3, weak_adj_Wi_SU2L_mix),cof(2, w_bar_Phi),cof(2, w_Phi))*B(nu)*Wi(mu,weak_adj_Wi_SU2L_mix)*Phi(w_Phi)*Phibar(w_bar_Phi)-1/6*g1*g(cof(2, w_bar_QL_weak_fund_id),cof(2, w_QL_weak_fund_id))*g(cof(3, fl_bar_QL_generation_id),cof(3, fl_QL_generation_id))*g(cof(3, c_bar_QL_color_fund_id),cof(3, c_QL_color_fund_id))*gamma(bis(4, i_bar_QL_U1Y),bis(4, i_QL_U1Y),mink(4, mu))*B(mu)*QL(i_QL_U1Y,w_QL_weak_fund_id,fl_QL_generation_id,c_QL_color_fund_id)*QLbar(i_bar_QL_U1Y,w_bar_QL_weak_fund_id,fl_bar_QL_generation_id,c_bar_QL_color_fund_id)+1/2*g1*g(cof(2, w_bar_LL_weak_fund_id),cof(2, w_LL_weak_fund_id))*g(cof(3, fl_bar_LL_generation_id),cof(3, fl_LL_generation_id))*gamma(bis(4, i_bar_LL_U1Y),bis(4, i_LL_U1Y),mink(4, mu))*B(mu)*LL(i_LL_U1Y,w_LL_weak_fund_id,fl_LL_generation

compiled terms : 1/2*gw*vev*PartialD(GPbar,mu_canon_1)*W(mu_canon_1)+1/2*gw*vev*PartialD(GP,mu_canon_1)*Wbar(mu_canon_1)+1𝑖*gw*sw*ghA*g(mink(4, mu_canon_1),mink(4, mu_canon_2))*PartialD(ghWpbar,mu_canon_2)*W(mu_canon_1)-1𝑖*gw*sw*ghA*g(mink(4, mu_canon_1),mink(4, mu_canon_2))*PartialD(ghWmbar,mu_canon_2)*Wbar(mu_canon_1)+1𝑖*gw*sw*ghWp*g(mink(4, mu_canon_1),mink(4, mu_canon_2))*PartialD(ghAbar,mu_canon_2)*Wbar(mu_canon_1)-1𝑖*gw*sw*ghWp*g(mink(4, mu_canon_1),mink(4, mu_canon_2))*PartialD(ghWpbar,mu_canon_2)*A(mu_canon_1)-1𝑖*gw*sw*ghWm*g(mink(4, mu_canon_1),mink(4, mu_canon_2))*PartialD(ghAbar,mu_canon_2)*W(mu_canon_1)+1𝑖*gw*sw*ghWm*g(mink(4, mu_canon_1),mink(4, mu_canon_2))*PartialD(ghWmbar,mu_canon_2)*A(mu_canon_1)-1𝑖/4*gw*sw*g(mink(4, mu_canon_1),mink(4, mu_canon_4))*PartialD(Wbar(mu_canon_2),mu_canon_4)*W(mu_canon_1)*A(mu_canon_2)+1𝑖/4*gw*sw*g(mink(4, mu_canon_1),mink(4, mu_canon_4))*PartialD(Wbar(mu_canon_2),mu_canon_4)*W(mu_canon_2)*A(mu_canon_1)+1𝑖/4*gw*sw*g(mink(4, mu_canon_1),mink

final terms : gs*g(mink(4, mu_canon_1),mink(4, mu_canon_2))*f(coad(8, a_canon_1),coad(8, a_canon_3),coad(8, a_canon_2))*PartialD(ghGbar(a_canon_1),mu_canon_2)*G(mu_canon_1,a_canon_3)*ghG(a_canon_2)-1/4*gs*g(mink(4, mu_canon_1),mink(4, mu_canon_4))*f(coad(8, a_canon_3),coad(8, a_canon_1),coad(8, a_canon_2))*PartialD(G(mu_canon_2,a_canon_3),mu_canon_4)*G(mu_canon_1,a_canon_1)*G(mu_canon_2,a_canon_2)-1/4*gs*g(mink(4, mu_canon_2),mink(4, mu_canon_4))*f(coad(8, a_canon_1),coad(8, a_canon_2),coad(8, a_canon_3))*PartialD(G(mu_canon_1,a_canon_1),mu_canon_4)*G(mu_canon_1,a_canon_3)*G(mu_canon_2,a_canon_2)+1/4*gs*g(mink(4, mu_canon_2),mink(4, mu_canon_4))*f(coad(8, a_canon_3),coad(8, a_canon_1),coad(8, a_canon_2))*PartialD(G(mu_canon_1,a_canon_3),mu_canon_4)*G(mu_canon_1,a_canon_1)*G(mu_canon_2,a_canon_2)+1/4*gs*g(mink(4, mu_canon_3),mink(4, mu_canon_4))*f(coad(8, a_canon_1),coad(8, a_canon_2),coad(8, a_canon_3))*PartialD(G(mu_canon_1,a_canon_1),mu_canon_4)*G(mu_canon_1,a_canon_2)*G(mu_canon_3,a

gauge-basis compiled terms : 91
weak-component terms       : 203
physical compiled terms    : 306
compact vertex signatures : 123
expanded arity counts      : {2: 30, 3: 139, 4: 32}
source model validation    : True
compiled validation        : True


### Higgs-only pipeline check

The same compilation chain can be inspected on the smaller Higgs kinetic and
potential sector alone.



$ \mathcal L_\Phi =
(D_\mu\Phi)^\dagger(D^\mu\Phi)
+\lambda v^2\Phi_i\Phi_i^\dagger
-\lambda\Phi_i\Phi_j\Phi_i^\dagger\Phi_j^\dagger .
$

$
\begin{aligned}
\mathcal L_{\Phi}^{\mathrm{gauge}}
&=
\frac12 g_1 g_W \eta_{\mu\nu}
T^a_{ij}
B^\mu W^{a\nu}
\Phi_j \Phi_i^\dagger
+
\frac12 g_1 g_W \eta_{\mu\nu}
T^a_{ij}
B^\nu W^{a\mu}
\Phi_j \Phi_i^\dagger
\\
&\quad
+\frac{i}{2}g_1\delta_{ij}
(\partial_\mu\Phi_j)B^\mu\Phi_i^\dagger
-\frac{i}{2}g_1\delta_{ij}
(\partial_\mu\Phi_i^\dagger)B^\mu\Phi_j
\\
&\quad
+i g_W T^a_{ij}
(\partial_\mu\Phi_j)W^{a\mu}\Phi_i^\dagger
-i g_W T^a_{ij}
(\partial_\mu\Phi_i^\dagger)W^{a\mu}\Phi_j
\\
&\quad
-\lambda \Phi_i\Phi_j\Phi_i^\dagger\Phi_j^\dagger
+\lambda v^2\Phi_i\Phi_i^\dagger
\\
&\quad
+\frac14 g_1^2\eta_{\mu\nu}\delta_{ij}
B^\mu B^\nu\Phi_j\Phi_i^\dagger
\\
&\quad
+g_W^2\eta_{\mu\nu}
T^a_{ik}T^b_{kj}
W^{a\mu}W^{b\nu}
\Phi_j\Phi_i^\dagger
\\
&\quad
+\delta_{ij}
(\partial_\mu\Phi_j)(\partial^\mu\Phi_i^\dagger).
\end{aligned}
$

$
\begin{aligned}
\mathcal L_{\Phi}^{\mathrm{weak\;expanded}}
&=
\frac14 g_1g_W\eta_{\mu\nu}
B^\mu W^{1\nu}\Phi_1\Phi_2^\dagger
+
\frac14 g_1g_W\eta_{\mu\nu}
B^\mu W^{1\nu}\Phi_2\Phi_1^\dagger
\\
&\quad
+\frac{i}{4}g_1g_W\eta_{\mu\nu}
B^\mu W^{2\nu}\Phi_1\Phi_2^\dagger
-\frac{i}{4}g_1g_W\eta_{\mu\nu}
B^\mu W^{2\nu}\Phi_2\Phi_1^\dagger
\\
&\quad
+\frac14 g_1g_W\eta_{\mu\nu}
B^\mu W^{3\nu}\Phi_1\Phi_1^\dagger
-\frac14 g_1g_W\eta_{\mu\nu}
B^\mu W^{3\nu}\Phi_2\Phi_2^\dagger
\\
&\quad
+\frac14 g_1g_W\eta_{\mu\nu}
B^\nu W^{1\mu}\Phi_1\Phi_2^\dagger
+
\frac14 g_1g_W\eta_{\mu\nu}
B^\nu W^{1\mu}\Phi_2\Phi_1^\dagger
\\
&\quad
+\frac{i}{4}g_1g_W\eta_{\mu\nu}
B^\nu W^{2\mu}\Phi_1\Phi_2^\dagger
-\frac{i}{4}g_1g_W\eta_{\mu\nu}
B^\nu W^{2\mu}\Phi_2\Phi_1^\dagger
\\
&\quad
+\frac14 g_1g_W\eta_{\mu\nu}
B^\nu W^{3\mu}\Phi_1\Phi_1^\dagger
-\frac14 g_1g_W\eta_{\mu\nu}
B^\nu W^{3\mu}\Phi_2\Phi_2^\dagger
\\
&\quad
+\frac{i}{2}g_1(\partial_\mu\Phi_1)B^\mu\Phi_1^\dagger
+\frac{i}{2}g_1(\partial_\mu\Phi_2)B^\mu\Phi_2^\dagger
\\
&\quad
-\frac{i}{2}g_1(\partial_\mu\Phi_1^\dagger)B^\mu\Phi_1
-\frac{i}{2}g_1(\partial_\mu\Phi_2^\dagger)B^\mu\Phi_2
\\
&\quad
+\frac{i}{2}g_W(\partial_\mu\Phi_1)W^{1\mu}\Phi_2^\dagger
-\frac12 g_W(\partial_\mu\Phi_1)W^{2\mu}\Phi_2^\dagger
+\frac{i}{2}g_W(\partial_\mu\Phi_1)W^{3\mu}\Phi_1^\dagger
\\
&\quad
+\frac{i}{2}g_W(\partial_\mu\Phi_2)W^{1\mu}\Phi_1^\dagger
+\frac12 g_W(\partial_\mu\Phi_2)W^{2\mu}\Phi_1^\dagger
-\frac{i}{2}g_W(\partial_\mu\Phi_2)W^{3\mu}\Phi_2^\dagger
\\
&\quad
-\frac{i}{2}g_W(\partial_\mu\Phi_1^\dagger)W^{1\mu}\Phi_2
-\frac12 g_W(\partial_\mu\Phi_1^\dagger)W^{2\mu}\Phi_2
-\frac{i}{2}g_W(\partial_\mu\Phi_1^\dagger)W^{3\mu}\Phi_1
\\
&\quad
-\frac{i}{2}g_W(\partial_\mu\Phi_2^\dagger)W^{1\mu}\Phi_1
+\frac12 g_W(\partial_\mu\Phi_2^\dagger)W^{2\mu}\Phi_1
+\frac{i}{2}g_W(\partial_\mu\Phi_2^\dagger)W^{3\mu}\Phi_2
\\
&\quad
-2\lambda\Phi_1\Phi_2\Phi_1^\dagger\Phi_2^\dagger
+\lambda v^2\Phi_1\Phi_1^\dagger
+\lambda v^2\Phi_2\Phi_2^\dagger
\\
&\quad
-\lambda\Phi_1^2(\Phi_1^\dagger)^2
-\lambda\Phi_2^2(\Phi_2^\dagger)^2
\\
&\quad
+(\partial_\mu\Phi_1)(\partial^\mu\Phi_1^\dagger)
+(\partial_\mu\Phi_2)(\partial^\mu\Phi_2^\dagger)
\\
&\quad
+\frac14 g_1^2\eta_{\mu\nu}B^\mu B^\nu
\left(
\Phi_1\Phi_1^\dagger+\Phi_2\Phi_2^\dagger
\right)
\\
&\quad
+\frac14 g_W^2\eta_{\mu\nu}
W^{a\mu}W^{a\nu}
\left(
\Phi_1\Phi_1^\dagger+\Phi_2\Phi_2^\dagger
\right)
+\cdots .
\end{aligned}
$

$
\begin{aligned}
\mathcal L_{\Phi}^{\mathrm{final}}
&=
-2\lambda vHG^-G^+
-\lambda vH(G^0)^2
-\lambda vH^3
-\lambda v^2H^2
+\frac14\lambda v^4
\\
&\quad
-\lambda H^2G^-G^+
-\frac12\lambda H^2(G^0)^2
-\frac14\lambda H^4
-\lambda(G^0)^2G^-G^+
-\frac14\lambda(G^0)^4
-\lambda(G^-)^2(G^+)^2
\\
&\quad
+\frac{ev}{2s_W}(\partial_\mu G^-)W^{+\mu}
+\frac{ev}{2s_W}(\partial_\mu G^+)W^{-\mu}
\\
&\quad
+\frac{i e^2v}{2s_W}
G^-W^+_\nu A^\nu
-\frac{i e^2v}{2s_W}
G^+W^-_\nu A^\nu
\\
&\quad
+\frac{e^2v}{2s_W^2}
H W^-_\mu W^{+\mu}
\\
&\quad
+i eG^-(\partial_\mu G^+)A^\mu
-i eG^+(\partial_\mu G^-)A^\mu
\\
&\quad
+\frac{ev}{2s_Wc_W}(\partial_\mu G^0)Z^\mu
+\frac{e^2v^2}{8s_W^2c_W^2}Z_\mu Z^\mu
+\frac{e^2v^2}{4s_W^2}W^-_\mu W^{+\mu}
\\
&\quad
+\frac{e}{2s_W}H(\partial_\mu G^-)W^{+\mu}
+\frac{e}{2s_W}H(\partial_\mu G^+)W^{-\mu}
\\
&\quad
+\frac{i e}{2s_W}G^0(\partial_\mu G^-)W^{+\mu}
-\frac{i e}{2s_W}G^0(\partial_\mu G^+)W^{-\mu}
\\
&\quad
-\frac{e}{2s_W}G^-(\partial_\mu H)W^{+\mu}
-\frac{e}{2s_W}G^+(\partial_\mu H)W^{-\mu}
\\
&\quad
-\frac{i e}{2s_W}G^-(\partial_\mu G^0)W^{+\mu}
+\frac{i e}{2s_W}G^+(\partial_\mu G^0)W^{-\mu}
\\
&\quad
+(\partial_\mu G^-)(\partial^\mu G^+)
+\frac12(\partial_\mu H)(\partial^\mu H)
+\frac12(\partial_\mu G^0)(\partial^\mu G^0)
\\
&\quad
+\frac{e^2}{2s_W^2}G^-G^+W^-_\mu W^{+\mu}
+\frac{e^2}{4s_W^2}H^2W^-_\mu W^{+\mu}
+\frac{e^2}{4s_W^2}(G^0)^2W^-_\mu W^{+\mu}
\\
&\quad
+e^2G^-G^+A_\mu A^\mu
+\frac{e^2}{8s_W^2c_W^2}H^2Z_\mu Z^\mu
+\frac{e^2}{8s_W^2c_W^2}(G^0)^2Z_\mu Z^\mu
\\
&\quad
+\cdots .
\end{aligned}
$

In [11]:
LHiggs_small = (
    DC(Phi.bar, mu) * DC(Phi, mu)
    + lam * vev**2 * Phi.bar * Phi
    - lam * Phi.bar * Phi * Phi.bar * Phi
)

higgs_source_model = Model(
    LHiggs_small,
    name=f"{MODEL_NAME} Higgs-only gauge basis",
    gauge_groups=gauge_groups,
    fields=source_fields,
    parameters=all_parameters,
)

higgs_L_gauge_basis = higgs_source_model.lagrangian()

higgs_L_weak_components = higgs_L_gauge_basis.expand_index_components(
    WEAK_FUND_INDEX,
    WEAK_ADJ_INDEX,
    tensor_components=weak_tensor_components,
)
higgs_L_physical = (
    higgs_L_weak_components
    .transform_fields(
        *transformations,
        repeat=False,
        real_symbols=real_symbols,
    )
    .simplify_parameter_identities()
)
higgs_L = express_in_physical_couplings(higgs_L_physical)
print("gauge-basis terms :", higgs_L_gauge_basis.to_symbolica())
print("physical terms :", higgs_L_weak_components.to_symbolica())
print("compiled terms :", higgs_L_physical.to_symbolica())
print("final terms :", higgs_L.to_symbolica())

higgs_source_report = higgs_source_model.validate()
higgs_compiled_report = higgs_L.validate()

print("gauge-basis compiled terms :", len(higgs_L_gauge_basis.terms))
print("weak-component terms       :", len(higgs_L_weak_components.terms))
print("physical compiled terms    :", len(higgs_L.terms))
print("source model validation    :", higgs_source_report.ok)
print("compiled validation        :", higgs_compiled_report.ok)

assert higgs_source_report.ok
assert higgs_compiled_report.ok


gauge-basis terms : 1/2*g1*gw*g(mink(4, mu),mink(4, nu))*t(coad(3, weak_adj_Wi_SU2L_mix),cof(2, w_bar_Phi),cof(2, w_Phi))*B(mu)*Wi(nu,weak_adj_Wi_SU2L_mix)*Phi(w_Phi)*Phibar(w_bar_Phi)+1/2*g1*gw*g(mink(4, mu),mink(4, nu))*t(coad(3, weak_adj_Wi_SU2L_mix),cof(2, w_bar_Phi),cof(2, w_Phi))*B(nu)*Wi(mu,weak_adj_Wi_SU2L_mix)*Phi(w_Phi)*Phibar(w_bar_Phi)+1𝑖/2*g1*g(cof(2, w_bar_Phi_weak_fund_id),cof(2, w_Phi_weak_fund_id))*PartialD(Phi(w_Phi_weak_fund_id),mu)*B(mu)*Phibar(w_bar_Phi_weak_fund_id)-1𝑖/2*g1*g(cof(2, w_bar_Phi_weak_fund_id),cof(2, w_Phi_weak_fund_id))*PartialD(Phibar(w_bar_Phi_weak_fund_id),mu)*B(mu)*Phi(w_Phi_weak_fund_id)+1𝑖*gw*t(coad(3, weak_adj_Wi_SU2L_1),cof(2, w_bar_Phi),cof(2, w_Phi))*PartialD(Phi(w_Phi),mu)*Wi(mu,weak_adj_Wi_SU2L_1)*Phibar(w_bar_Phi)-1𝑖*gw*t(coad(3, weak_adj_Wi_SU2L_1),cof(2, w_bar_Phi),cof(2, w_Phi))*PartialD(Phibar(w_bar_Phi),mu)*Wi(mu,weak_adj_Wi_SU2L_1)*Phi(w_Phi)-lam*Phi(w_decl_1)*Phi(w_decl_2)*Phibar(w_decl_1)*Phibar(w_decl_2)+lam*vev^2*Phi(w_decl_1)*

## 9. Inspect before extracting a rule

`vertex_signatures` reports which field multisets exist. It can filter by
arity, exact signature, contained fields, or sector. `matching_terms` exposes
the compiled monomials that contribute to a requested vertex.


In [12]:
report = L.vertex_report()
hww_signature = L.vertex_signatures(signature=(H, W.bar, W))
hww_terms = L.matching_terms(H, W.bar, W)

print("compiled terms              :", report.total_terms)
print("grouped signatures          :", report.total_signatures)
print("three-point signatures      :", len(L.vertex_signatures(arity=3)))
print("four-point signatures       :", len(L.vertex_signatures(arity=4)))
print("ghost-containing signatures :", sum(
    any(name.startswith("gh") for name in signature.names)
    for signature in L.vertex_signatures()
))
print("exact H W- W+ signatures    :", len(hww_signature))
print("terms contributing to HWW   :", len(hww_terms))
print()
for signature in L.vertex_signatures(arity=3)[:12]:
    print(f"{signature.names!s:<34} terms={signature.term_count}")


compiled terms              : 306
grouped signatures          : 123
three-point signatures      : 69
four-point signatures       : 32
ghost-containing signatures : 31
exact H W- W+ signatures    : 1
terms contributing to HWW   : 1

('G', 'G', 'G')                    terms=4
('G0', 'G0', 'Z')                  terms=2
('G0', 'GP', 'W.bar')              terms=2
('GP', 'W.bar', 'A')               terms=1
('GP', 'W.bar', 'Z')               terms=2
('GP.bar', 'A', 'W')               terms=1
('GP.bar', 'G0', 'W')              terms=2
('GP.bar', 'GP', 'A')              terms=2
('GP.bar', 'GP', 'H')              terms=2
('GP.bar', 'GP', 'Z')              terms=2
('GP.bar', 'H', 'W')               terms=2
('GP.bar', 'W', 'Z')               terms=2


## 10. Calculate representative Feynman rules




In [13]:
representative_rules = (
    ("Gamma(W-, W+, A)", (W.bar, W, A), {}),
    ("Gamma(H, W-, W+)", (H, W.bar, W), {}),
    ("Gamma(H, H, H)", (H, H, H), {}),
    ("Gamma(lbar, l, A)", (l.bar, l, A), {}),
    ("Gamma(ebar, e, A)", (l.class_members[0].bar, l.class_members[0], A), {"flavor_expand": True}),
    ("Gamma(tbar, b, W+)", (uq.class_members[2].bar, dq.class_members[2], W), {"flavor_expand": True}),
    ("Gamma(ubar, u, G)", (uq.bar, uq, G), {}),
    ("Gamma(ghGbar, ghG, G)", (ghG.bar, ghG, G), {}),
)

for label, external_fields, options in representative_rules:
    rule = L.feynman_rule(
        *external_fields,
        simplify=True,
        include_delta=False,
        **options,
    )
    print("=" * 116)
    print_rule(label, rule)


Gamma(W-, W+, A)
  -1𝑖*ee*pcomp(q1,mu2)*g(mink(4,mu1),mink(4,mu3))+-1𝑖*ee*pcomp(q2,mu3)*g(mink(4,mu1),mink(4,mu2))+-1𝑖*ee*pcomp(q3,mu1)*g(mink(4,mu2),mink(4,mu3))+1𝑖*ee*pcomp(q1,mu3)*g(mink(4,mu1),mink(4,mu2))+1𝑖*ee*pcomp(q2,mu1)*g(mink(4,mu2),mink(4,mu3))+1𝑖*ee*pcomp(q3,mu2)*g(mink(4,mu1),mink(4,mu3))
Gamma(H, W-, W+)
  1𝑖/2*ee^2*sw^(-2)*vev*g(mink(4,mu2),mink(4,mu3))
Gamma(H, H, H)
  -6𝑖*lam*vev
Gamma(lbar, l, A)
  -1𝑖/2*ee*g(cof(3,fl1),cof(3,fl2))*gamma(bis(4,i1),bis(4,i2),mink(4,mu3))+1𝑖/2*ee*g(cof(3,fl1),cof(3,fl2))*gamma(bis(4,bis_canon_1),bis(4,bis_canon_2),mink(4,mu3))*gamma5(bis(4,bis_canon_2),bis(4,i2))*gamma5(bis(4,i1),bis(4,bis_canon_1))
Gamma(ebar, e, A)
  -1𝑖/2*ee*gamma(bis(4,i1),bis(4,i2),mink(4,mu3))+1𝑖/2*ee*gamma(bis(4,bis_canon_1),bis(4,bis_canon_2),mink(4,mu3))*gamma5(bis(4,bis_canon_2),bis(4,i2))*gamma5(bis(4,i1),bis(4,bis_canon_1))
Gamma(tbar, b, W+)
  (1/2)^(1/2)*-1𝑖/4*CKM33*ee*sw^(-1)*g(cof(3,c1),cof(3,c2))*gamma(bis(4,bis_canon_1),bis(4,bis_canon_2),mink(4,mu3))

## 11. All nonzero three- and four-point FeynPy rules

For this Standard Model configuration there are 171
candidates, 8 cancellations, and **163 nonzero rules**.



In [14]:
from models.SM.feynrules_comparison import standard_model_feynrules_name_aliases

aliases = standard_model_feynrules_name_aliases(fields)
nonzero_rules = []
zero_candidates = []

for signature in L.vertex_signatures(flavor_expand=True):
    if signature.arity not in (3, 4):
        continue
    key = "|".join(sorted(aliases.get(name, name) for name in signature.names))
    expression = L.feynman_rule(
        *signature.fields,
        simplify=True,
        include_delta=False,
        flavor_expand=True,
    )
    if expression.to_canonical_string() == "0":
        zero_candidates.append(key)
    else:
        nonzero_rules.append((key, signature.names, expression))

nonzero_rules.sort(key=lambda item: item[0])
zero_candidates.sort()

print("candidate signatures:", len(nonzero_rules) + len(zero_candidates))
print("identically zero     :", len(zero_candidates))
print("nonzero rules        :", len(nonzero_rules))
print("zero candidates      :", zero_candidates)
print()

for number, (key, native_names, expression) in enumerate(nonzero_rules, start=1):
    print("=" * 116)
    print(f"[{number:03d}/163] {key}")
    print("FeynPy fields:", native_names)
    print_rule("FeynPy rule:", expression)

assert len(zero_candidates) == 8
assert len(nonzero_rules) == 163


candidate signatures: 171
identically zero     : 8
nonzero rules        : 163
zero candidates      : ['G0|G0|G0|H', 'G0|G0|Z', 'G0|GP|GPbar|H', 'G0|H|H', 'G0|H|H|H', 'G0|H|W|Wbar', 'G0|H|Z|Z', 'H|H|Z']

[001/163] A|A|GP|GPbar
FeynPy fields: ('GP.bar', 'GP', 'A', 'A')
FeynPy rule:
  2𝑖*ee^2*g(mink(4,mu3),mink(4,mu4))
[002/163] A|A|W|Wbar
FeynPy fields: ('W.bar', 'A', 'W', 'A')
FeynPy rule:
  -2𝑖*ee^2*g(mink(4,mu1),mink(4,mu3))*g(mink(4,mu2),mink(4,mu4))+1𝑖*ee^2*g(mink(4,mu1),mink(4,mu2))*g(mink(4,mu3),mink(4,mu4))+1𝑖*ee^2*g(mink(4,mu1),mink(4,mu4))*g(mink(4,mu2),mink(4,mu3))
[003/163] A|G0|GPbar|W
FeynPy fields: ('GP.bar', 'G0', 'A', 'W')
FeynPy rule:
  -1𝑖/2*ee^2*sw^(-1)*g(mink(4,mu3),mink(4,mu4))
[004/163] A|G0|GP|Wbar
FeynPy fields: ('G0', 'GP', 'W.bar', 'A')
FeynPy rule:
  -1𝑖/2*ee^2*sw^(-1)*g(mink(4,mu3),mink(4,mu4))
[005/163] A|GPbar|H|W
FeynPy fields: ('GP.bar', 'H', 'A', 'W')
FeynPy rule:
  -1/2*ee^2*sw^(-1)*g(mink(4,mu3),mink(4,mu4))
[006/163] A|GPbar|W
FeynPy fields: ('GP.bar'

## 12. Original FeynRules `SM.fr` output


In [15]:
reference_path = (
    MODEL_DIR / "reference" / "feynrules"
    / "sm_vertices_FeynRules.json"
)
original_feynrules = json.loads(reference_path.read_text(encoding="utf-8"))

print("reference file:", reference_path.relative_to(ROOT))
print("original FeynRules rules:", len(original_feynrules))
print()
for number, vertex in enumerate(original_feynrules, start=1):
    print("=" * 116)
    print(f"[{number:03d}/163] id={vertex['id']}  key={vertex['key']}")
    print("FeynRules field order:", vertex["fields"])
    print("FeynRules legs       :", vertex["legs"])
    print("FeynRules rule       :")
    print("  " + vertex["rule"])

assert len(original_feynrules) == 163


reference file: models/SM/reference/feynrules/sm_vertices_FeynRules.json
original FeynRules rules: 163

[001/163] id=10  key=A|A|GP|GPbar
FeynRules field order: ['A', 'A', 'GP', 'GPbar']
FeynRules legs       : ['{A, 1}', '{A, 2}', '{GP, 3}', '{GPbar, 4}']
FeynRules rule       :
  (2*I)*ee^2*ME[Index[Lorentz, Ext[1]], Index[Lorentz, Ext[2]]]
[002/163] id=92  key=A|A|W|Wbar
FeynRules field order: ['A', 'A', 'W', 'Wbar']
FeynRules legs       : ['{A, 1}', '{A, 2}', '{W, 3}', '{Wbar, 4}']
FeynRules rule       :
  I*ee^2*ME[Index[Lorentz, Ext[1]], Index[Lorentz, Ext[4]]]*ME[Index[Lorentz, Ext[2]], Index[Lorentz, Ext[3]]] + I*ee^2*ME[Index[Lorentz, Ext[1]], Index[Lorentz, Ext[3]]]*ME[Index[Lorentz, Ext[2]], Index[Lorentz, Ext[4]]] - (2*I)*ee^2*ME[Index[Lorentz, Ext[1]], Index[Lorentz, Ext[2]]]*ME[Index[Lorentz, Ext[3]], Index[Lorentz, Ext[4]]]
[003/163] id=119  key=A|b|bbar
FeynRules field order: ['bbar', 'b', 'A']
FeynRules legs       : ['{bbar, 1}', '{b, 2}', '{A, 3}']
FeynRules rule       